# Fazenda Pamplona: posicionamento RTK e PPP

Este caderno reorganizado documenta a **visualização dos dados já digitalizados** da figura de posicionamento na página 8 de `2024AGU_SH24B-07_Marquez.pdf`. Os valores aproximam traços rasterizados; não são observações originais do receptor. A origem é o PDF e os painéis recortados estão em `KEYNOTE/inputs/digitization/source_assets/`.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "pyproject.toml").exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Abra o Jupyter dentro do checkout do projeto.")
sys.path.insert(0, str(PROJECT_ROOT / "src")) if str(PROJECT_ROOT / "src") not in sys.path else None
from sbgea_keynote.paths import input_path, output_path

DATA_FILE = output_path("data", "digitized_positioning_error", "digitized_positioning_error.csv")
SOURCE_PDF = input_path("references", "2024AGU_SH24B-07_Marquez.pdf")
SOURCE_ASSETS = input_path("digitization", "source_assets")
FIGURES = output_path("figures", "keynote", "posicionamento_fazenda_pamplona_pt.png")

data = pd.read_csv(DATA_FILE)
data["timestamp_BRT"] = pd.to_datetime(data["timestamp_BRT"])
print(f"PDF-fonte encontrado: {SOURCE_PDF.exists()}")
print(f"Registros digitalizados: {len(data):,}")
display(data.groupby("series").agg(
    amostras=("error_2D_m", "size"),
    erro_mediano_m=("error_2D_m", "median"),
    erro_maximo_m=("error_2D_m", "max"),
).round(3))

## Controle visual das fontes

Os recortes permitem comparar os traços digitalizados com a imagem publicada. A envoltória no CSV registra a faixa de pixels associada ao traço e ajuda a comunicar que a precisão é limitada pela resolução da figura.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.5), facecolor="white")
for ax, filename, title in zip(
    axes,
    ["source_page_08.png", "rtk_panel_source.png", "ppp_panel_source.png"],
    ["Página 8 do PDF", "Painel RTK rover–base", "Painel PPP"],
):
    path = SOURCE_ASSETS / filename
    if path.exists():
        ax.imshow(plt.imread(path))
        ax.set_title(title, color="#082F73")
    else:
        ax.text(.5, .5, f"Fonte ausente: {filename}", ha="center", va="center")
    ax.set_axis_off()
fig.tight_layout()
plt.show()

## Leitura dos dados

O eixo temporal está em horário de Brasília (BRT). O preenchimento mostra a faixa vertical do traço rasterizado, não um intervalo estatístico de confiança do receptor. A incerteza estimada de digitalização é distinta da incerteza física/instrumental dos dados originais.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.2), facecolor="white")
colors = {"RTK Rover–Base": "#2686D0", "PPP Receiver": "#EC8135"}
for name, group in data.groupby("series", sort=False):
    group = group.sort_values("timestamp_BRT")
    color = colors.get(name, "#70AE54")
    ax.fill_between(group["timestamp_BRT"], group["raster_envelope_min_m"],
                    group["raster_envelope_max_m"], color=color, alpha=.16, linewidth=0)
    ax.plot(group["timestamp_BRT"], group["error_2D_m"], color=color, lw=1.7, label=name)
ax.set(xlabel="Horário (BRT)", ylabel="Erro horizontal 2D (m)")
ax.set_title("Digitalização dos traços publicados", loc="left", color="#082F73", weight="bold")
ax.grid(color="#DCE6ED", linewidth=.7)
ax.legend(frameon=False)
fig.autofmt_xdate()
fig.tight_layout()
plt.show()

## Figura final em português

Esta célula recria a figura pronta para apresentação a partir dos CSVs mantidos em `KEYNOTE/outputs/data/`. Ela não altera nem reinterpreta os pontos digitalizados.

In [ ]:
NAVY, BLUE, ORANGE, GRAY, WHITE = "#082F73", "#2686D0", "#EC8135", "#707070", "#FFFFFF"
fig, axes = plt.subplots(1, 2, figsize=(12.0, 6.1), facecolor=WHITE)
fig.subplots_adjust(left=.09, right=.98, bottom=.16, top=.80, wspace=.32)
fig.suptitle("RESULTADOS — Erro de posicionamento na Fazenda Pamplona",
             fontsize=19, fontweight="bold", color=GRAY, y=.95)
for ax, name, title, color in [
    (axes[0], "RTK Rover–Base", "RTK: rover em relação à base", BLUE),
    (axes[1], "PPP Receiver", "Posicionamento PPP", ORANGE),
]:
    g = data.loc[data["series"] == name].sort_values("timestamp_BRT")
    ax.fill_between(g["timestamp_BRT"], g["raster_envelope_min_m"],
                    g["raster_envelope_max_m"], color=color, alpha=.18, linewidth=0,
                    label="faixa do traço na imagem")
    ax.plot(g["timestamp_BRT"], g["error_2D_m"], color=color, lw=1.8,
            label="centro digitalizado")
    ax.set_title(title, color=NAVY, fontsize=12, loc="left")
    ax.set_ylabel("Erro horizontal 2D (m)")
    ax.set_xlabel("Data e hora (BRT)")
    ax.grid(color="#DCE6ED", linewidth=.7)
    ax.legend(frameon=False, fontsize=8.5)
    ax.tick_params(axis="x", labelrotation=25)
fig.text(.09, .035,
         "Digitalização aproximada de gráficos rasterizados; a faixa representa pixels do traço, não erro estatístico.",
         fontsize=8.5, color=GRAY)
fig.savefig(FIGURES, dpi=200, bbox_inches="tight", facecolor=WHITE)
plt.show()
print(f"Figura salva em: {FIGURES}")